# Полный OCR: ориентация текста 0°/180°

Этот ноутбук самостоятельно создаёт `submission.csv` для всех 20 000 тестовых кропов.
Оба локальных компонента — `PP-LCNet_x1_0_textline_ori` и
`eslav_PP-OCRv5_mobile_rec` — запускаются **для каждого изображения** в двух
ориентациях; отбор только неуверенных случаев отсутствует. Вероятность равна
`sigmoid(logit(p_base) + 20 × (score_180 − score_0))`. Коэффициент 20 выбран
на псевдоразмеченной валидации HierText, без разметки теста Avito. На ней
получено 0.99408876 по 1−Brier; балл Stepik для этого варианта неизвестен.

Ноутбук автоматически выберет `gpu:0`, если CUDA-сборка Paddle и GPU доступны;
иначе он запустится на CPU и будет заметно медленнее. Для Windows/Python 3.13
установите [официальный CUDA 12.9 wheel PaddlePaddle 3.3.0](https://www.paddlepaddle.org.cn/documentation/docs/en/install/pip/windows-pip.html),
затем `pip install paddleocr==3.7.0 numpy==2.3.5 Pillow==12.1.0`.
Запускайте из корня проекта с `models/` и `test_data/` (либо `test.zip`);
при необходимости используйте `AVITO_PROJECT_DIR` и `AVITO_TEST_PATH`.
Веса уже лежат в `models/`; внешние API и тестовые метки не используются.
После запуска выводится время каждой модели и полное время. Запуск этого
ноутбука заменит `submission.csv`, созданный быстрым ноутбуком.

In [ ]:
from __future__ import annotations

import csv
import io
import math
import os
import site
import time
import zipfile
from pathlib import Path

import sys
# These notebooks use Paddle only; a broken optional Torch installation can block PaddleX.
sys.modules['torch'] = None


# On Windows, NVIDIA pip wheels keep CUDA DLLs in these directories.
CUDA_DLL_HANDLES = []
if os.name == "nt":
    for site_root in site.getsitepackages():
        for library in ("cudnn", "cublas", "cudart", "cufft", "curand", "cusolver", "cusparse"):
            dll_dir = Path(site_root) / "nvidia" / library / "bin"
            if dll_dir.is_dir():
                CUDA_DLL_HANDLES.append(os.add_dll_directory(str(dll_dir)))
                os.environ["PATH"] = str(dll_dir) + os.pathsep + os.environ["PATH"]

import numpy as np
import paddle
from PIL import Image
from paddleocr import TextLineOrientationClassification, TextRecognition

PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", Path.cwd())).expanduser().resolve()
ORIENTATION_MODEL_DIR = PROJECT_DIR / "models" / "PP-LCNet_x1_0_textline_ori_infer"
OCR_MODEL_DIR = PROJECT_DIR / "models" / "eslav_PP-OCRv5_mobile_rec_infer"
OUTPUT_CSV = PROJECT_DIR / "submission.csv"
BATCH_PAIRS = 32
OCR_COEFFICIENT = 20.0

for model_dir in (ORIENTATION_MODEL_DIR, OCR_MODEL_DIR):
    if not model_dir.is_dir():
        raise FileNotFoundError(f"Model weights not found: {model_dir}")
gpu_ready = paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
DEVICE = "gpu:0" if gpu_ready else "cpu"
print(f"Project: {PROJECT_DIR}")
print(f"Paddle {paddle.__version__}; inference device: {DEVICE}")

In [ ]:
EXPECTED_ROWS = 20_000

def find_test_data(project_dir: Path) -> Path:
    explicit = os.environ.get("AVITO_TEST_PATH")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    for path in (project_dir / "test_data", project_dir / "test.zip"):
        if path.exists():
            return path
    raise FileNotFoundError("Set AVITO_TEST_PATH or put test_data/test.zip next to the notebook")

class TestData:
    """Read test images in exactly the order of sample_submission.csv."""

    def __init__(self, path: Path):
        self.archive = None
        self.image_dir = None
        if path.is_file():
            self.archive = zipfile.ZipFile(path)
            sample = self.archive.read("sample_submission.csv")
        elif path.is_dir():
            if (path / "test" / "images").is_dir():
                self.image_dir = path / "test" / "images"
                sample_path = path / "sample_submission.csv"
            elif (path / "images").is_dir():
                self.image_dir = path / "images"
                sample_path = path / "sample_submission.csv"
                if not sample_path.is_file():
                    sample_path = path.parent / "sample_submission.csv"
            else:
                raise FileNotFoundError(f"No test/images directory in {path}")
            sample = sample_path.read_bytes()
        else:
            raise FileNotFoundError(path)
        reader = csv.DictReader(io.StringIO(sample.decode("utf-8-sig")))
        if reader.fieldnames != ["image_id", "p_180"]:
            raise ValueError(f"Unexpected sample columns: {reader.fieldnames}")
        self.image_ids = [row["image_id"] for row in reader]
        if len(self.image_ids) != EXPECTED_ROWS or len(set(self.image_ids)) != EXPECTED_ROWS:
            raise ValueError("Expected 20,000 unique image IDs")

    def __enter__(self):
        return self

    def __exit__(self, *_):
        if self.archive is not None:
            self.archive.close()

    def read_image(self, image_id: str) -> bytes:
        name = f"{image_id}.png"
        if self.archive is not None:
            return self.archive.read(f"test/images/{name}")
        return (self.image_dir / name).read_bytes()

    def check_images(self) -> None:
        if self.archive is not None:
            available = set(self.archive.namelist())
            missing = [name for name in self.image_ids if f"test/images/{name}.png" not in available]
        else:
            missing = [name for name in self.image_ids if not (self.image_dir / f"{name}.png").is_file()]
        if missing:
            raise FileNotFoundError(f"Missing test image: {missing[0]}")

In [ ]:
PROBABILITY_LIMIT = 1e-6

def probability_180(result) -> float:
    label = result["label_names"][0]
    score = float(result["scores"][0])
    if label not in ("0_degree", "180_degree") or not math.isfinite(score):
        raise ValueError(f"Unexpected classifier output: {label}, {score}")
    if not 0 <= score <= 1:
        raise ValueError(f"Invalid classifier confidence: {score}")
    return score if label == "180_degree" else 1 - score

def logit(value: float) -> float:
    value = min(1 - PROBABILITY_LIMIT, max(PROBABILITY_LIMIT, value))
    return math.log(value / (1 - value))

def sigmoid(value: float) -> float:
    if value >= 0:
        return 1 / (1 + math.exp(-value))
    exp_value = math.exp(value)
    return exp_value / (1 + exp_value)

def baseline_probability(original: float, rotated: float) -> float:
    return sigmoid((logit(original) - logit(rotated)) / 2)

In [ ]:
total_start = time.perf_counter()
with TestData(find_test_data(PROJECT_DIR)) as test:
    test.check_images()

    start = time.perf_counter()
    orientation_model = TextLineOrientationClassification(
        model_name="PP-LCNet_x1_0_textline_ori",
        model_dir=str(ORIENTATION_MODEL_DIR),
        device=DEVICE,
    )
    orientation_load_seconds = time.perf_counter() - start

    start = time.perf_counter()
    ocr_model = TextRecognition(
        model_name="eslav_PP-OCRv5_mobile_rec",
        model_dir=str(OCR_MODEL_DIR),
        device=DEVICE,
    )
    ocr_load_seconds = time.perf_counter() - start

    image_seconds = 0.0
    orientation_seconds = 0.0
    ocr_seconds = 0.0
    rows = []
    for start_index in range(0, len(test.image_ids), BATCH_PAIRS):
        ids = test.image_ids[start_index:start_index + BATCH_PAIRS]

        start = time.perf_counter()
        images = []
        for image_id in ids:
            with Image.open(io.BytesIO(test.read_image(image_id))) as source:
                image = source.convert("RGB")
            images.append(np.asarray(image))
            images.append(np.asarray(image.transpose(Image.Transpose.ROTATE_180)))
        image_seconds += time.perf_counter() - start

        start = time.perf_counter()
        orientation = orientation_model.predict(images, batch_size=2 * BATCH_PAIRS)
        orientation_seconds += time.perf_counter() - start
        start = time.perf_counter()
        recognition = ocr_model.predict(images, batch_size=2 * BATCH_PAIRS)
        ocr_seconds += time.perf_counter() - start
        if len(orientation) != len(images) or len(recognition) != len(images):
            raise RuntimeError("Unexpected number of model predictions")

        for position, image_id in enumerate(ids):
            p0 = probability_180(orientation[2 * position])
            p180 = probability_180(orientation[2 * position + 1])
            base = baseline_probability(p0, p180)
            score0 = float(recognition[2 * position]["rec_score"])
            score180 = float(recognition[2 * position + 1]["rec_score"])
            if not all(math.isfinite(x) and 0 <= x <= 1 for x in (score0, score180)):
                raise ValueError(f"Invalid OCR score for {image_id}")
            probability = sigmoid(logit(base) + OCR_COEFFICIENT * (score180 - score0))
            rows.append((image_id, probability))

        done = start_index + len(ids)
        if done % 1024 == 0 or done == len(test.image_ids):
            print(f"Processed {done}/{len(test.image_ids)}", flush=True)

    if [image_id for image_id, _ in rows] != test.image_ids:
        raise ValueError("Output IDs differ from sample_submission.csv")

start = time.perf_counter()
temporary = OUTPUT_CSV.with_suffix(".csv.tmp")
with temporary.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(("image_id", "p_180"))
    writer.writerows((image_id, f"{p:.8f}") for image_id, p in rows)
os.replace(temporary, OUTPUT_CSV)
csv_seconds = time.perf_counter() - start
total_seconds = time.perf_counter() - total_start

print(f"Saved {len(rows)} predictions to {OUTPUT_CSV}")
print(f"Orientation model load: {orientation_load_seconds:.2f} s")
print(f"OCR model load: {ocr_load_seconds:.2f} s")
print(f"Read and rotate: {image_seconds:.2f} s")
print(f"Orientation inference: {orientation_seconds:.2f} s")
print(f"OCR inference: {ocr_seconds:.2f} s")
print(f"CSV write: {csv_seconds:.2f} s")
print(f"Total: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Throughput: {len(rows) / total_seconds:.1f} crops/s")